# IFEval analysis

In [1]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# configure for retina display
%config InlineBackend.figure_format = "retina"

# set the style and context
sns.set_theme(context='talk', style='whitegrid', palette='muted')

# DataFrame pre-processing

In [3]:
# HUGGINGFACE_MODEL = "Llama-3-8B"
HUGGINGFACE_MODEL = "Llama-3.2-1B"

PIPELINES = {
    # "disciple": f"{HUGGINGFACE_MODEL} (SMC w/ DisCIPL)",
    "disciple_smc": f"DisCIPL ({HUGGINGFACE_MODEL} w/ SMC)",
    "disciple_iid": f"DisCIPL ({HUGGINGFACE_MODEL} w/ IID)",
    # "unconditional_sampling": f"Unconditional Sampling",
    "huggingface": f"{HUGGINGFACE_MODEL} (Sampling)",
    "huggingface_beam_search": f"{HUGGINGFACE_MODEL} (Beam Search)",
    "gpt-4o-mini": f"GPT-4o-mini",
    "gpt-4o": f"GPT-4o",
}

PALETTE = {
    # PIPELINES["disciple"]: "tab:orange",
    PIPELINES["disciple_smc"]: "tab:orange",
    PIPELINES["disciple_iid"]: "tab:green",
    # PIPELINES["unconditional_sampling"]: "tab:blue",
    PIPELINES["huggingface"]: "tab:pink",
    PIPELINES["huggingface_beam_search"]: "tab:red",
    PIPELINES["gpt-4o-mini"]: "tab:purple",
    PIPELINES["gpt-4o"]: "tab:cyan",
}

TIMESTAMP_TO_PIPELINES = {
    "2025-01-26-23-15-32": [
        "huggingface",
        "huggingface_beam_search",
        "gpt-4o-mini",
        "gpt-4o",
    ],  # baselines
    # "2025-01-26-23-08-28": ["disciple_smc", "disciple_iid"],
    "2025-01-27-12-23-04": ["disciple_smc", "disciple_iid"],
}

In [ ]:
# load the data
df_list = []
existing_pipelines = set()
for timestamp, pipelines in TIMESTAMP_TO_PIPELINES.items():
    print(f"Loading results from {timestamp}: {pipelines}")
    _df = pd.read_json(f"results/{timestamp}/results.json")

    # warn on duplicate pipelines
    new_pipelines = set(_df["pipeline_name"].unique())
    duplicates = new_pipelines.intersection(existing_pipelines)
    if duplicates:
        print(f"WARNING: {duplicates} already in existing pipelines")

    # filter the pipelines
    _df = _df[_df["pipeline_name"].isin(pipelines)]
    df_list.append(_df)
df = pd.concat(df_list).reset_index(drop=True)

In [5]:
# Unpack task dict into columns
df_task = df["task"].apply(pd.Series)
# Rename columns
df_task = df_task.rename(columns={"prompt": "task_prompt"})
# Drop columns
if "evaluators" in df_task.columns:
    df_task = df_task.drop(columns=["evaluators"])
if "examples" in df_task.columns:
    df_task = df_task.drop(columns=["examples"])

# Unpack evaluate_results dict into columns
df_evaluate_results = df["evaluate_results"].apply(pd.Series)
# Rename columns
df_evaluate_results = df_evaluate_results.rename(
    columns={metric: f"valid_{metric}" for metric in df_evaluate_results.columns}
)

# Combine dataframes
df = pd.concat(
    [df_task, df, df_evaluate_results],
    axis=1,
)
df = df.drop(columns=["task", "evaluate_results"])

df["valid_true"] = (df["valid_strict"] == True)

# df["error"] = df["error"].astype(str)
if "error" not in df.columns:
    df["error"] = None
df["error_true"] = ~df["error"].isnull()

# Convert task_types (list) to string
df["task_type"] = df["task_types"].apply(lambda x: ", ".join(x))


df["method"] = df["pipeline_name"].map(PIPELINES)
df["method"] = pd.Categorical(
    df["method"],
    categories=[v for k, v in PIPELINES.items() if k in df["pipeline_name"].unique()],
)

# Reorder columns (put "task_id", "task_types", "task_prompt" first, then the rest)
COL_ORDER = ["task_id", "task_types", "task_prompt", "pipeline_name", "method", "N"]
COL_ORDER += [col for col in df.columns if col not in COL_ORDER]
df = df[COL_ORDER]

# Filter out pipelines that are not in the list
df = df[df["pipeline_name"].isin(PIPELINES.keys())].reset_index(drop=True)

# Only include results from final attempts
df_all_attempts = df.copy()
df = df[df["is_final_attempt"] == True].reset_index(drop=True)

In [ ]:
df

In [ ]:
# create a "long" copy of the dataframe where tasks with multiple task_types are duplicated
# df_long should be used in plots that facet by task_type
df_long = df.copy()

del df_long["task_type"]
df_long["task_types_orig"] = df_long["task_types"]
df_long = df_long.explode("task_types").rename(
    columns={"task_types": "task_type", "task_types_orig": "task_types"}
)
df_long = df_long.reset_index(drop=True)
df_long["task_level"] = df_long["task_type"].apply(lambda x: x.split(":")[0])

print(len(df_long))
print(df_long["task_type"].unique())
print(df_long["task_level"].unique())

## Data validation checks

In [ ]:
N_TASKS = df["task_id"].nunique()
print(f"Number of tasks: {N_TASKS}")

assert not df["N"].isnull().any()

method_counts = df['method'].value_counts()

if method_counts.nunique() != 1:
    raise ValueError(f"Methods have different number of rows: {method_counts.to_dict()}")

# for N, group in df.groupby("N"):
#     if (group["method"].value_counts() / N != N_TASKS).all():
#         raise ValueError(f"Number of tasks is not consistent for N={N}")

In [ ]:
df.groupby(["N", "method"]).size().unstack().plot(kind="bar", stacked=False, color=[PALETTE[method] for method in df["method"].unique()])

In [ ]:
df.groupby(["method"])["error_true"].value_counts(normalize=True)

# Eval metrics

In [ ]:
ax = sns.lineplot(data=df, x="N", y="valid_true", hue="method", palette=PALETTE, marker="o")

ax.set_xlabel("Samples")
ax.set_ylabel("Valid")

# Set x-axis labels to correspond exactly to the values of the points
x_values = df["N"].unique()
# ax.set_xscale("log", base=2)
ax.set_xticks(x_values)
ax.set_xticklabels(x_values)

# Move the legend outside the plot
ax.legend(title="Method", loc="center left", bbox_to_anchor=(1, 0.5))

In [ ]:
df["valid_breakdown"] = df[["valid_true", "error_true"]].apply(
    lambda x: (
        "Error" if x["error_true"] else ("Valid" if x["valid_true"] else "Invalid")
    ),
    axis=1,
)

df["N_value"] = df["N"].astype(str)

palette = {
    "Valid": "tab:green",
    "Invalid": "tab:red",
    "Error": "tab:grey",
}

g = sns.displot(
    data=df,
    x="N_value",
    hue="valid_breakdown",
    col="method",
    stat="proportion",
    discrete=True,
    multiple="fill",
    palette=palette,
    hue_order=list(palette.keys()),
)

In [ ]:
sns.catplot(kind="bar", col="task_type", data=df_long, x="N", y="valid_true", hue="method", palette=PALETTE)

In [ ]:
g = sns.FacetGrid(
    df_long,
    col="task_type",
    # col="task_type_n",
    # row="task_level",
    hue="method",
    palette=PALETTE,
    aspect=1.0,
    sharex=False,
)

g.map(sns.lineplot, "N", "valid_true", marker="o")
g.add_legend(title="Method")

# Set x-axis labels to correspond exactly to the values of the points
for ax in g.axes.flat:
    x_values = df["N"].unique()

    ax.set_xscale("log", base=2)
    ax.set_xticks(x_values)
    ax.set_xticklabels(x_values)

    # Remove "task_name = " from the subtitle
    ax.set_title(
        ax.get_title()
        .replace("task_type = ", "")
        .replace("task_type_n = ", "")
        .replace("task_level = ", "")
        .replace(" | ", "_")
    )

    # Set y-axis limits
    # ax.set_ylim(None, 1)

g.set_axis_labels("Samples")
g.set_ylabels("Valid")

In [ ]:
sns.displot(df, x="weight", hue="method", kind="kde", palette=PALETTE)

In [ ]:
g = sns.boxplot(df, x="valid_true", y="weight", hue="method", palette=PALETTE, showfliers=False)
g.set(ylim=(-1000, None))

In [ ]:
g = sns.boxplot(
    df, hue="valid_true", y="weight", x="method", showfliers=False
)
g.set(ylim=(-1000, None))
g.set_xticklabels(g.get_xticklabels(), rotation=90);

In [ ]:
import numpy as np
df["neginf_weight"] = (df["weight"] == -np.inf)
df["neginf_weight"].value_counts()

In [19]:
# # Apply weight corrections
# df.loc[df["pipeline_name"] == "huggingface", "weight"] /= df.loc[
#     df["pipeline_name"] == "huggingface", "token_count"
# ]

# df.loc[df["pipeline_name"] == "huggingface_beam_search", "weight"] /= df.loc[
#     df["pipeline_name"] == "huggingface_beam_search", "token_count"
# ]

# df.loc[df["pipeline_name"] == "gpt-4o", "weight"] /= df.loc[
#     df["pipeline_name"] == "gpt-4o", "token_count"
# ]

# df.loc[df["pipeline_name"] == "gpt-4o-mini", "weight"] /= df.loc[
#     df["pipeline_name"] == "gpt-4o-mini", "token_count"
# ]

# sns.displot(df, x="weight", hue="method", kind="kde", palette=PALETTE)

## Pass@k

In [ ]:
def compute_pass_at_k(df, k_values=range(1, 11)):
    """For each group of size k, pass@k is True if any rows in the group are valid and False otherwise."""
    GROUP_COLS = ["task_type", "task_id", "method", "N"]
    df_list = []
    for (task_type, task_id, method, N), group in df.groupby(GROUP_COLS)[
        df.columns.tolist()
    ]:
        for k in k_values:
            # if len(group) < k:
            #     break
            if N < k:
                continue

            if group["error_true"].all():
                pass_at_k = False
                error_true = True
            else:
                pass_at_k = group.nlargest(k, "weight", keep="all")["valid_true"].any() # weighted pass@k
                # pass_at_k = group.sample(n=k, replace=False)["valid_true"].any()  # random pass@k
                error_true = False

            df_list.append(
                {
                    "task_id": task_id,
                    "task_type": task_type,
                    "method": method,
                    "N": N,
                    "k": k,
                    f"pass@k": pass_at_k,
                    "error_true": error_true,
                }
            )
    return pd.DataFrame(df_list)


df_pass_at_k = compute_pass_at_k(df, k_values=df["N"].unique().tolist())

df_pass_at_k_long = compute_pass_at_k(df_long, k_values=df_long["N"].unique().tolist())

In [ ]:
df_pass_at_k_counts = df_pass_at_k.groupby(["method", "N", "k"]).count()

_df_missing_tasks = df_pass_at_k_counts[df_pass_at_k_counts.ne(N_TASKS).any(axis=1)]
assert _df_missing_tasks.empty, _df_missing_tasks

print(f"Each row has {N_TASKS} tasks.")

### pass@1

In [ ]:
df_pass_at_1 = df_pass_at_k[df_pass_at_k["k"] == 1]
df_pass_at_1

In [ ]:
g = sns.FacetGrid(
    df_pass_at_k_long[df_pass_at_k_long["k"] == 1], col="task_type", hue="method", palette=PALETTE, aspect=1.0
)

g.map(sns.lineplot,
      "N",
      "pass@k",
    #   marker="o"
      )
g.add_legend(title="Method")

# Set x-axis labels to correspond exactly to the values of the points
for ax in g.axes.flat:
    x_values = df["N"].unique()

    # ax.set_xscale("log", base=2)
    ax.set_xticks(x_values)
    ax.set_xticklabels(x_values)

    # Remove "task_name = " from the subtitle
    ax.set_title(ax.get_title().replace("task_type = ", ""))

    # Set y-axis limits
    # ax.set_ylim(None, 1)

g.set_axis_labels("Samples")
g.set_ylabels(f"Pass@1")

In [ ]:
ax = sns.lineplot(data=df_pass_at_1, x="N", y="pass@k", hue="method", palette=PALETTE, marker="o")

ax.set_xlabel("Samples")
ax.set_ylabel("Pass@1")

# Set x-axis labels to correspond exactly to the values of the points
x_values = df["N"].unique()
# ax.set_xscale("log", base=2)
ax.set_xticks(x_values)
ax.set_xticklabels(x_values)

# Move the legend outside the plot
ax.legend(title="Method", loc="center left", bbox_to_anchor=(1, 0.5))

In [ ]:
g = sns.FacetGrid(
    df_pass_at_k_long[df_pass_at_k_long["k"] == 1], col="task_type", hue="method", palette=PALETTE, aspect=1.0
)

g.map(sns.lineplot, "N", "error_true", marker="o")
# g.map(sns.barplot, "N", "error_true", order=sorted(df_pass_at_1["N"].unique()))
g.add_legend(title="Method")

# # Set x-axis labels to correspond exactly to the values of the points
for ax in g.axes.flat:

    # Remove "task_name = " from the subtitle
    ax.set_title(ax.get_title().replace("task_type = ", ""))

    # Set the x axis to df_pass_at_1["N"].unique()
    x_values = df_pass_at_1["N"].unique()
    ax.set_xscale("log", base=2)
    ax.set_xticks(x_values)
    ax.set_xticklabels(x_values)

    # Set y-axis limits
    ax.set_ylim(None, 1)

g.set_axis_labels("Samples")
g.set_ylabels(f"Runtime Error @ 1")

### Pass@k

In [ ]:
g = sns.relplot(
    data=df_pass_at_k_long,
    x="N",
    y="pass@k",
    row="k",
    style="k",
    markers=False,
    dashes=False,
    hue="method",
    col="task_type",
    kind="line",
    palette=PALETTE,
    errorbar=None,
)

# Set x-axis labels to correspond exactly to the values of the points
for ax in g.axes.flat:
    x_values = df["N"].unique()

    ax.set_xscale("log", base=2)
    ax.set_xticks(x_values)
    ax.set_xticklabels(x_values)

    # Remove "task_name = " from the subtitle
    ax.set_title(ax.get_title().replace("task_type = ", ""))

    # Set y-axis limits
    # ax.set_ylim(None, 1)

In [ ]:
g = sns.relplot(
    data=df_pass_at_k,
    x="N",
    y="pass@k",
    # row="task_type",
    col="k",
    style="k",
    markers=True,
    dashes=False,
    hue="method",
    kind="line",
    palette=PALETTE,
    errorbar=None,
    facet_kws={"sharex": False},
)

# Set x-axis labels to correspond exactly to the values of the points
for ax in g.axes.flat:
    x_values = df["N"].unique()

    ax.set_xscale("log", base=2)
    ax.set_xticks(x_values)
    ax.set_xticklabels(x_values)

    # Set y-axis limits
    # ax.set_ylim(None, 1)

In [ ]:
g = sns.relplot(
    data=df_pass_at_k,
    x="k",
    y="pass@k",
    # row="task_type",
    # col="k",
    style="N",
    # size="N",
    dashes=False,
    hue="method",
    kind="line",
    palette=PALETTE,
    errorbar=None,
    markers=True,
    facet_kws={"sharex": False},
    aspect=1.5,
)

# Set x-axis labels to correspond exactly to the values of the points
for ax in g.axes.flat:
    x_values = df_pass_at_k["k"].unique()

    # ax.set_xscale("log", base=2)
    ax.set_xticks(x_values)
    ax.set_xticklabels(x_values)

    # Set y-axis limits
    # ax.set_ylim(None, 1)

In [ ]:
g = sns.relplot(
    data=df_pass_at_k,
    x="N",
    y="pass@k",
    # row="task_type",
    # col="k",
    style="k",
    size="k",
    dashes=False,
    hue="method",
    kind="line",
    palette=PALETTE,
    errorbar=None,
    markers=True,
    facet_kws={"sharex": False},
    aspect=1.5,
)

# Set x-axis labels to correspond exactly to the values of the points
for ax in g.axes.flat:
    x_values = df_pass_at_k["N"].unique()

    # ax.set_xscale("log", base=2)
    ax.set_xticks(x_values)
    ax.set_xticklabels(x_values)

    # Set y-axis limits
    # ax.set_ylim(None, 1)

In [ ]:
g = sns.relplot(
    data=df_pass_at_k[df_pass_at_k["N"] == df_pass_at_k["k"]],
    x="k",
    y="pass@k",
    hue="method",
    kind="line",
    palette=PALETTE,
    markers=True,
    facet_kws={"sharex": False},
    aspect=1.5,
)

# Set x-axis labels to correspond exactly to the values of the points
for ax in g.axes.flat:
    x_values = df_pass_at_k["k"].unique()

    ax.set_xscale("log", base=2)
    ax.set_xticks(x_values)
    ax.set_xticklabels(x_values)

    # Set y-axis limits
    # ax.set_ylim(None, 1)

plt.title("Pass@k for N = k")

In [ ]:
from evaluations.plotting import plot_pass_at_k

fig = plot_pass_at_k(
    df_pass_at_k[df_pass_at_k["N"] == df_pass_at_k["k"]],
    # df_pass_at_1,
    dataset="IFeval",
    subtasks="all",
    palette=PALETTE,
    fig_width=18,
)

In [ ]:
g = sns.barplot(
    data=df_pass_at_k[df_pass_at_k["N"] == df_pass_at_k["k"]],
    x="k",
    y="error_true",
    hue="method",
    palette=PALETTE,
)

# Move the legend outside the plot
g.legend(title="Method", loc="center left", bbox_to_anchor=(1, 0.5))

plt.title("Error rate for N = k")

In [ ]:
k_max = df["N"].max()

df_pass_at_k_max = df_pass_at_k[df_pass_at_k["k"] == k_max]

g = sns.barplot(
    data=df_pass_at_k_max,
    x="task_type",
    y="pass@k",
    hue="method",
    palette=PALETTE,
)

# Move the legend outside the plot
g.legend(title="Method", loc='center left', bbox_to_anchor=(1, 0.5))

g.set_ylabel(f"Pass@{k_max}")

# Error analysis

In [ ]:
df.groupby(["method"])["error_true"].value_counts(normalize=True)

In [ ]:
df["error"].isnull().value_counts(normalize=True)

In [35]:
# PIPELINE_NAME = "disciple_iid"
PIPELINE_NAME = "disciple_smc"

In [ ]:
rows_per_task_type = (
    df[df["pipeline_name"] == PIPELINE_NAME]
    .groupby("task_type")["task_id"]
    .count()
    .to_frame()
)
rows_per_task_type

In [ ]:
error_counts = (
    df_long[df_long["pipeline_name"] == PIPELINE_NAME].copy()
    .groupby("task_type")["error"]
    .value_counts(dropna=False, normalize=False, ascending=False)
    .to_frame(name="count")
)

# Replace NaN with "No error"
error_counts = error_counts.reset_index()
error_counts["error"] = error_counts["error"].fillna("No error")
error_counts = error_counts.set_index(["task_type", "error"])

# Join with rows_per_task_type
error_counts = error_counts.join(rows_per_task_type, on="task_type")

# Divide "count" by "rows_per_task_type"
error_counts["count"] = error_counts["count"] / error_counts["task_id"]

# Drop the "task_id" column
error_counts = error_counts.drop(columns=["task_id"])

error_counts

In [ ]:
TOP_N_ERRORS = 20

df_pipeline = df_long[df_long["pipeline_name"] == PIPELINE_NAME].copy()
df_pipeline.loc[:, "error"] = df_pipeline.loc[:, "error"].fillna("No error")

# Get the top N most frequent errors
n_unique_errors = df_pipeline["error"].nunique()
top_errors = df_pipeline["error"].value_counts().nlargest(min(TOP_N_ERRORS - 1, n_unique_errors)).index

# Map all other errors to "Other"
df_pipeline.loc[~df_pipeline["error"].isin(top_errors), "error"] = "Other"

# Order the hue by error frequency
error_order = df_pipeline["error"].value_counts().index

sns.displot(
    data=df_pipeline,
    y="task_type",
    hue="error",
    hue_order=error_order,
    stat="proportion",
    discrete=True,
    multiple="fill",
    shrink=0.8,
    # height=8,
    palette="tab20",
)

# plt.xticks(rotation=90)
plt.title(f"Error distribution for {PIPELINE_NAME}");

In [ ]:
df_error_wide = (
    df[df["pipeline_name"] == PIPELINE_NAME]
    .groupby("task_type")["error"]
    .value_counts(dropna=False, normalize=False, ascending=True)
    .reset_index()
    .set_index("error")
    .pivot(columns="task_type", values="count")
)
df_error_wide = df_error_wide.sort_index(ascending=False)

g = df_error_wide.plot(kind="barh", stacked=True, legend=False)
g.legend(loc='center left', bbox_to_anchor=(1, 0.5))

In [40]:
df_error = df[~df.error.isna()]

In [ ]:
plot = sns.countplot(data=df_error, y="task_type")

# Evaluation of text

In [42]:
import numpy as np

# df["text_length"] = df["text"].apply(lambda x: len(x) if x is not None else 0)
df["text_length"] = df["text"].apply(lambda x: len(x) if x is not None else np.nan)


In [ ]:
sns.displot(
    data=df,
    kind="kde",
    x="text_length",
    hue="method",
    palette=PALETTE
)

In [ ]:
RANDOM_STATE = 888
RANDOM_TASK_IDS = (
    df.groupby("task_type").sample(n=1, random_state=RANDOM_STATE)["task_id"].tolist()
)
print(RANDOM_TASK_IDS)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    # display(
    #     df.groupby(["task_id"]).sample(n=3, random_state=0)[
    #         ["task_type", "task_id", "method", "task_prompt", "text"]
    #     ]
    # )
    display(
        df[
            (df["N"] == df["N"].max())
            # & (df["method"] == PIPELINES[PIPELINE_NAME])
            # & (df["valid_true"])
            & df["task_id"].isin(RANDOM_TASK_IDS)
        ]
        .groupby(["task_id", "method"])
        .sample(n=1, random_state=RANDOM_STATE)
        .sort_values(["task_type", "method", "weight"], ascending=[True, True, False])[
            [
                "task_type",
                "task_id",
                "method",
                "task_prompt",
                "text",
                "weight",
                "valid_true",
            ]
        ]
    )

In [ ]:
RANDOM_STATE = 888
RANDOM_TASK_IDS = (
    df.groupby("task_type").sample(n=1, random_state=RANDOM_STATE)["task_id"].tolist()
)
print(RANDOM_TASK_IDS)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    # display(
    #     df.groupby(["task_id"]).sample(n=3, random_state=0)[
    #         ["task_type", "task_id", "method", "task_prompt", "text"]
    #     ]
    # )
    display(
        df[df["task_id"].isin(RANDOM_TASK_IDS) & (df["N"] == df["N"].max())]dfdfdfddfdfdf
        .sort_values(["task_type", "method", "weight"], ascending=[True, True, False])
        [
            ["task_type", "task_id", "method", "task_prompt", "text", "weight", "valid_true"]
        ]
    )

# Debugging